In [1]:
# ==============================================================================
# CELL 1: DJANGO BOOTSTRAP — chạy cell này TRƯỚC TIÊN, mọi cell khác phụ thuộc
# ==============================================================================
import sys
import os

os.chdir("/home/duoc/app")  # chuyển về thư mục project
sys.path.insert(0, "/home/duoc/app")  # đảm bảo Python tìm thấy config/

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings.local")

import django

django.setup()  # Bắt buộc — khởi tạo App Registry trước khi dùng ORM

print("Django setup OK —", django.get_version())
print("Settings module  :", os.environ["DJANGO_SETTINGS_MODULE"])

import nest_asyncio

nest_asyncio.apply()

Project root: /home/duoc/app/agent_os


In [2]:
# Cell 2 — Imports

import asyncio

# Runtime engine — cung cấp LiteEmbeddingEngine
from system.runtime.runtime_engine import build_runtime

# Knowledge layer
from system.knowledge.knowledge_engine import KnowledgeEngine
from system.knowledge.ingestion_service import IngestionService
from system.knowledge.retrieval_service import RetrievalService
from system.knowledge.schemas import IngestRequest, RetrievalRequest

print("Imports OK")

11:56:37 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
11:56:37 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


ModuleNotFoundError: No module named 'system.knowledge'

In [ ]:
import sys

sys.path.append("/app")
sys.path.append("/app/ai_stack")

# ============================================
# JUPYTER MOCK TEST
# KHÔNG DÙNG DJANGO DB
# KHÔNG DÙNG CELERY
# TEST:
# - loader
# - ingest
# - vector db
# ============================================
import sys
import pathlib

# Trỏ về root của project (thư mục chứa manage.py)
PROJECT_ROOT = pathlib.Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
from pathlib import Path
from types import SimpleNamespace

from agent_os.system.knowledge.schemas import (
    IngestRequest,
)

from agent_os.system.runtime.llm_model import (
    create_isolated_services,
)


# ============================================
# MOCK DOCUMENT
# ============================================

mock_doc = SimpleNamespace(
    id=999,
    title="Jupyter Test",
    status="pending",
    chunk_count=0,
    vector_doc_id=None,
)

# ============================================
# TEST FUNCTION
# ============================================


async def test_ingest_file(
    file_path_str: str,
):

    print("=" * 60)
    print("START MOCK INGEST TEST")
    print("=" * 60)

    services = None

    try:
        # -------------------------------------------------
        # FILE
        # -------------------------------------------------

        print("\n[1] VALIDATE FILE")

        file_path = Path(file_path_str)

        print("PATH:", file_path)

        if not file_path.exists():
            raise FileNotFoundError(f"Missing file: {file_path}")

        print("FILE EXISTS OK")

        # -------------------------------------------------
        # SERVICES
        # -------------------------------------------------

        print("\n[2] CREATE SERVICES")

        services = await create_isolated_services()

        print(
            "RAG AVAILABLE:",
            services.rag_available,
        )

        if not services.rag_available:
            raise RuntimeError("KnowledgeEngine unavailable")

        # -------------------------------------------------
        # LOADER
        # -------------------------------------------------

        print("\n[3] LOAD DOCUMENT")

        loader = services.document_loader

        loaded_doc = loader.load_file(str(file_path))

        print("LOAD SUCCESS")

        # -------------------------------------------------
        # METADATA
        # -------------------------------------------------

        print("\n[4] ORIGINAL METADATA")

        print(loaded_doc.metadata)

        # -------------------------------------------------
        # ENRICH
        # -------------------------------------------------

        print("\n[5] ENRICH METADATA")

        loaded_doc.metadata.update(
            {
                "source_id": str(mock_doc.id),
                "source_title": mock_doc.title,
                "source_type": file_path.suffix.replace(".", ""),
            }
        )

        print(loaded_doc.metadata)

        # -------------------------------------------------
        # CONTENT
        # -------------------------------------------------

        print("\n[6] CONTENT PREVIEW")

        print(loaded_doc.text[:1500])

        # -------------------------------------------------
        # INGEST REQUEST
        # -------------------------------------------------

        print("\n[7] CREATE INGEST REQUEST")

        ingest_request = IngestRequest(
            text=loaded_doc.text,
            metadata=loaded_doc.metadata,
        )

        print("INGEST REQUEST OK")

        # -------------------------------------------------
        # INGEST
        # -------------------------------------------------

        print("\n[8] INGEST")

        result = await services.ingest.ingest(ingest_request)

        print("INGEST SUCCESS")

        print("\nVECTOR DOC ID:")
        print(result.doc_id)

        # -------------------------------------------------
        # MOCK SUCCESS
        # -------------------------------------------------

        mock_doc.status = "completed"
        mock_doc.chunk_count = 1
        mock_doc.vector_doc_id = result.doc_id

        print("\n[9] MOCK DB UPDATE")

        print(vars(mock_doc))

        print("\nSUCCESS")

    except Exception as exc:
        mock_doc.status = "failed"

        print("\nERROR")
        print(type(exc).__name__)
        print(exc)

        raise

    finally:
        print("\n[10] CLEANUP")

        try:
            if services and services.knowledge_engine:
                vector_store = services.knowledge_engine._vector_store

                if hasattr(
                    vector_store,
                    "_engine",
                ):
                    await vector_store._engine.dispose()

                    print("DB ENGINE DISPOSED")

        except Exception as e:
            print(
                "CLEANUP ERROR:",
                e,
            )

        print("\nDONE")

In [8]:
# ==============================================================================
# SỬA TRIỆT ĐỂ: TRUY XUẤT QUA _ENGINE ĐỂ FLUSH SẠCH VECTOR INDEX
# ==============================================================================
from agent_os.bootstrap import llm_model
from django.db import connection
from asgiref.sync import sync_to_async
from knowledge.models import DocumentNode, DocumentSource


async def clear_rag_force():
    # 1. Làm sạch Postgres Store trước (Rất quan trọng)
    print("1. Đang dọn dẹp các bảng dữ liệu trong Postgres...")
    await sync_to_async(DocumentNode.objects.all().delete)()
    await sync_to_async(DocumentSource.objects.all().delete)()

    # Truncate để reset hoàn toàn ID tự tăng và xóa liên kết embedding nếu có
    with connection.cursor() as cursor:
        cursor.execute("TRUNCATE TABLE knowledge_documentnode CASCADE;")
        cursor.execute("TRUNCATE TABLE knowledge_documentsource CASCADE;")
    print("   -> Đã làm sạch Postgres hoàn toàn.")

    # 2. Xử lý Vector Store qua tầng ẩn _engine
    svc = await llm_model()
    engine = svc.retrieval._engine
    print(f"2. Loại Engine tìm thấy: {type(engine).__name__}")

    # Duyệt tìm các thuộc tính lưu trữ bên trong _engine
    engine_attrs = dir(engine)

    # Tìm kiếm Vector Store nằm trong Index hoặc Retriever của LlamaIndex
    vector_store = None
    if hasattr(engine, "_index"):
        index_obj = getattr(engine, "_index")
        if hasattr(index_obj, "vector_store"):
            vector_store = index_obj.vector_store
    elif hasattr(engine, "_vector_store"):
        vector_store = getattr(engine, "_vector_store")
    elif hasattr(engine, "vector_store"):
        vector_store = getattr(engine, "vector_store")

    if vector_store:
        v_store_name = type(vector_store).__name__
        print(f"   -> Tìm thấy cấu trúc lưu trữ Vector: {v_store_name}")

        # Nếu là PGVector (Vector lưu luôn trong Postgres của bạn)
        if "postgres" in v_store_name.lower() or "pg" in v_store_name.lower():
            with connection.cursor() as cursor:
                # Quét sạch bảng lưu trữ vector mở rộng
                cursor.execute(
                    "DROP TABLE IF EXISTS data_documentnode_embedding CASCADE;"
                )
                cursor.execute("DROP TABLE IF EXISTS data_vector_store CASCADE;")
            print("   -> Đã clear bảng PGVector lưu trong Postgres.")

        # Nếu là ChromaDB / Qdrant nằm dưới tầng Vector Store
        elif hasattr(vector_store, "client"):
            client = vector_store.client
            client_name = type(client).__name__.lower()
            if "chroma" in client_name:
                for col in client.list_collections():
                    client.delete_collection(name=col.name)
            elif "qdrant" in client_name:
                for col in client.get_collections().collections:
                    client.delete_collection(collection_name=col.name)
            print("   -> Đã xóa sạch dữ liệu trên Vector Client độc lập.")
    else:
        print(
            "   -> Không thể can thiệp bằng code runtime do cấu trúc Engine đóng gói kín."
        )
        print("   -> CHUYỂN SANG PHƯƠNG ÁN 2 (XÓA VẬT LÝ) DƯỚI ĐÂY.")


await clear_rag_force()

1. Đang dọn dẹp các bảng dữ liệu trong Postgres...


SynchronousOnlyOperation: You cannot call this from an async context - use a thread or sync_to_async.

In [5]:
# 🎉 [SUCCESS] Toàn bộ hệ thống RAG đã được làm sạch thành công!
# CELL JUPYTER: HARD RESET KNOWLEDGE BASE (ĐÃ SỬA LỖI APP REGISTRY)
# ==============================================================================
import os
import sys

# 1. Định hình chính xác PATH cho dự án
sys.path.append("/app")
sys.path.append("/app/ai_stack")

# 2. Khởi tạo môi trường Django (Bắt buộc phải chạy trước khi import Model)
if not os.environ.get("DJANGO_SETTINGS_MODULE"):
    os.environ.setdefault(
        "DJANGO_SETTINGS_MODULE", "agent_os.system.config.settings"
    )  # Thay thế path settings nếu dự án của bạn nằm ở chỗ khác

import django

django.setup()

# 3. Tiến hành các import sau khi Django đã sẵn sàng
import asyncio
from asgiref.sync import sync_to_async
from django.db import transaction
from knowledge.models import DocumentSource
from agent_os.system.runtime.llm_model import create_isolated_services


async def hard_reset_rag_system():
    print("🧹 [START] Khởi động tiến trình quét sạch Knowledge Base...")

    # Khởi tạo cụm dịch vụ để lấy kết nối tới Vector Store
    services = await create_isolated_services()

    # 4. Xóa sạch Vector trong Vector Store
    if services and services.knowledge_engine:
        try:
            vector_store = services.knowledge_engine._vector_store

            if hasattr(vector_store, "clear"):
                vector_store.clear()
                print("   ✅ [Vector Store] Đã thực thi hàm clear().")
            elif hasattr(vector_store, "_engine") and vector_store._engine:
                from sqlalchemy import text

                async with vector_store._engine.begin() as conn:
                    await conn.execute(text("TRUNCATE TABLE data_vector_node CASCADE;"))
                print("   ✅ [Vector Store] SQL Truncate executed successfully.")
        except Exception as ve:
            print(f"   ⚠️ [Vector Store] Cảnh báo dọn dẹp: {ve}")

    # 5. Xóa sạch các bản ghi trong Django Database bằng Atomic Transaction
    @sync_to_async
    def _purge_django_orm_records():
        with transaction.atomic():
            total_records = DocumentSource.objects.all().count()
            if total_records > 0:
                DocumentSource.objects.all().delete()
            return total_records

    try:
        deleted_count = await _purge_django_orm_records()
        print(
            f"   ✅ [Django DB] Đã xóa vĩnh viễn {deleted_count} bản ghi DocumentSource."
        )
    except Exception as dbe:
        print(f"   ❌ [Django DB] Lỗi khi clear records: {dbe}")

    print("=" * 80)
    print("🎉 [SUCCESS] Toàn bộ hệ thống RAG đã được làm sạch thành công!")
    print("=" * 80)


# Kích hoạt chạy trực tiếp trên Jupyter Cell
await hard_reset_rag_system()

INFO 2026-05-18 18:12:34,765 llm_model 1408 133020630719360 [AgentServices] 🌀 Khởi tạo cụm Isolated Services dành riêng cho Celery Task...
ERROR 2026-05-18 18:12:34,766 llm_model 1408 133020630719360 [AgentServices] ❌ KnowledgeEngine unavailable
Traceback (most recent call last):
  File "/app/agent_os/system/runtime/llm_model.py", line 286, in _bootstrap
TypeError: KnowledgeEngine.create() missing 1 required keyword-only argument: 'docstore'


🧹 [START] Khởi động tiến trình quét sạch Knowledge Base...
   ✅ [Django DB] Đã xóa vĩnh viễn 31 bản ghi DocumentSource.
🎉 [SUCCESS] Toàn bộ hệ thống RAG đã được làm sạch thành công!


In [1]:
# ==============================================================================
# CELL 2: CONFIGURATION (UPDATED FOR MATRIX STRESS TEST V2)
# ==============================================================================
QUESTIONS_TO_TEST = [
    # Câu 1: Test khả năng tính toán logic/trích xuất số liệu ẩn (25% của 40%)
    "Tổng ngân sách dự kiến mà tập đoàn cấp cho nhãn hàng EcoCare là bao nhiêu và phân bổ cụ thể cho kênh Google Search & Display Network là bao nhiêu phần trăm tổng ngân sách?",
    # Câu 2: Test khả năng bẫy mốc thời gian (Thời điểm hỏi là tháng 7, nhưng tài liệu ghi tháng 9 mới áp dụng giá mới)
    "Vào ngày 20/07/2026, tôi có thể mua sản phẩm nước lau sàn EcoCare bản 1.5L trên hệ thống siêu thị WinMart với giá bao nhiêu?",
    # Câu 3: Test bẫy phân quyền theo hạn mức chi phí chính xác
    "Một bài viết giới thiệu sản phẩm mới EcoCare có chi phí media là 20 triệu VND thì ai là người có quyền duyệt cuối cùng để đăng tải?",
    # Câu 4: Test bẫy biến động chỉ số (145k so với gốc 135k là vượt > 15%, phải kích hoạt Kịch bản B chứ không phải Kịch bản A)
    "Nếu kênh KOL/KOC TikTok bị tăng Target CPA lên mức 145.000 VND liên tiếp trong 1 tuần, hệ thống sẽ xử lý thế nào và ai chịu trách nhiệm?",
    # Câu 5: Test so sánh dữ liệu đa tầng trong bảng thông số KPI
    "So sánh Target CPA mục tiêu của kênh TikTok Video Ads với Target CPA mục tiêu của kênh KOL/KOC & Booking Community của EcoCare trong chiến dịch Quý 3.",
    # Câu 6: Test bẫy thực thể trùng lặp (Tài liệu chỉ nói quy trình chung cho EcoCare và EcoHome, xem RAG có bốc được text không)
    "Quy trình duyệt bài đăng dưới 50 triệu của nhãn hàng EcoHome quy định ai là người trực tiếp phê duyệt?",
    # Câu 7: Test bẫy địa phương (Chiết khấu 5% cho Miền Trung từ tháng 9)
    "Từ tháng 9 năm 2026 trở đi, giá niêm yết của sản phẩm nước lau sàn EcoCare 1.5L khi bán tại hệ thống đại lý độc quyền ở Đà Nẵng là bao nhiêu?",
    # Câu 8: Test trích xuất chi phí cố định theo địa phương
    "Chiến dịch quảng cáo ngoài trời (OOH) của EcoCare tại các trạm xe buýt Đà Nẵng có ngân sách cố định là bao nhiêu?",
]

TOP_K = 5
SCORE_THRESHOLD = 0.60

print(f"Đã nạp {len(QUESTIONS_TO_TEST)} câu hỏi test hệ thống.")
print(f"Cấu hình: top_k={TOP_K} | score_threshold>={SCORE_THRESHOLD}")

Đã nạp 8 câu hỏi test hệ thống.
Cấu hình: top_k=5 | score_threshold>=0.6


In [ ]:
# ==============================================================================
# CELL: FULL INTEGRATED MATRIX STRESS TEST V2 (ISOLATED THREAD RUNTIME)
# ==============================================================================
import os
import sys
from typing import List
import asyncio
import threading

# 1. SETUP SYSTEM PATHS & DJANGO ENVIRONMENT
sys.path.insert(0, "/app")
sys.path.append("/app/ai_stack")

# Khai báo settings module trước khi import bất cứ thứ gì của Django
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings.local")

import django

django.setup()  # Khởi tạo App Registry bắt buộc trước khi dùng ORM

print("Django setup OK —", django.get_version())
print("Settings module  :", os.environ["DJANGO_SETTINGS_MODULE"])


# 2. MATRIX STRESS TEST CONFIGURATION
QUESTIONS_TO_TEST = [
    # Câu 1: Test khả năng tính toán logic/trích xuất số liệu ẩn (25% của 40%)
    "Tổng ngân sách dự kiến mà tập đoàn cấp cho nhãn hàng EcoCare là bao nhiêu và phân bổ cụ thể cho kênh Google Search & Display Network là bao nhiêu phần trăm tổng ngân sách?",
    # Câu 2: Test khả năng bẫy mốc thời gian (Thời điểm hỏi là tháng 7, nhưng tài liệu ghi tháng 9 mới áp dụng giá mới)
    "Vào ngày 20/07/2026, tôi có thể mua sản phẩm nước lau sàn EcoCare bản 1.5L trên hệ thống siêu thị WinMart với giá bao nhiêu?",
    # Câu 3: Test bẫy phân quyền theo hạn mức chi phí chính xác
    "Một bài viết giới thiệu sản phẩm mới EcoCare có chi phí media là 20 triệu VND thì ai là người có quyền duyệt cuối cùng để đăng tải?",
    # Câu 4: Test bẫy biến động chỉ số (145k so với gốc 135k là vượt > 15%, phải kích hoạt Kịch bản B chứ không phải Kịch bản A)
    "Nếu kênh KOL/KOC TikTok bị tăng Target CPA lên mức 145.000 VND liên tiếp trong 1 tuần, hệ thống sẽ xử lý thế nào và ai chịu trách nhiệm?",
    # Câu 5: Test so sánh dữ liệu đa tầng trong bảng thông số KPI
    "So sánh Target CPA mục tiêu của kênh TikTok Video Ads với Target CPA mục tiêu của kênh KOL/KOC & Booking Community của EcoCare trong chiến dịch Quý 3.",
    # Câu 6: Test bẫy thực thể trùng lặp (Tài liệu chỉ nói quy trình chung cho EcoCare và EcoHome, xem RAG có bốc được text không)
    "Quy trình duyệt bài đăng dưới 50 triệu của nhãn hàng EcoHome quy định ai là người trực tiếp phê duyệt?",
    # Câu 7: Test bẫy địa phương (Chiết khấu 5% cho Miền Trung từ tháng 9)
    "Từ tháng 9 năm 2026 trở đi, giá niêm yết của sản phẩm nước lau sàn EcoCare 1.5L khi bán tại hệ thống đại lý độc quyền ở Đà Nẵng là bao nhiêu?",
    # Câu 8: Test trích xuất chi phí cố định theo địa phương
    "Chiến dịch quảng cáo ngoài trời (OOH) của EcoCare tại các trạm xe buýt Đà Nẵng có ngân sách cố định là bao nhiêu?",
]

TOP_K = 5
SCORE_THRESHOLD = 0.60

print(f"Đã nạp {len(QUESTIONS_TO_TEST)} câu hỏi test hệ thống.")
print(f"Cấu hình: top_k={TOP_K} | score_threshold>={SCORE_THRESHOLD}")


# 3. HELPERS DEFINITION
from agent_os.knowledge.schemas import RetrievedChunk


def _detect_duplicates(chunks: List[RetrievedChunk]) -> List[int]:
    seen: set = set()
    dupes: List[int] = []
    for i, chunk in enumerate(chunks, 1):
        fp = " ".join(chunk.text.split())
        if fp in seen:
            dupes.append(i)
        else:
            seen.add(fp)
    return dupes


def _indent(text: str, prefix: str = "      ") -> str:
    return "\n".join(f"{prefix}{line}" for line in text.strip().splitlines())


def _preview_text(text: str, max_lines: int = 2) -> str:
    """Cắt ngắn text, chỉ giữ lại N dòng đầu tiên để log siêu gọn gàng."""
    lines = [line.strip() for line in (text or "").strip().splitlines() if line.strip()]
    if not lines:
        return "(trống)"
    preview = lines[:max_lines]
    result = " / ".join(preview)
    if len(lines) > max_lines:
        result += " ... [Xem tiếp trên Admin]"
    return result


def _parent_status(chunk: RetrievedChunk) -> str:
    if not chunk.parent_text:
        return "EMPTY"
    return "RESOLVED"


# ==============================================================================
# 4. REFACTORED JUPYTER-SAFE ASYNC EXECUTION FLOW
# ==============================================================================
import pandas as pd
import nest_asyncio
from asgiref.sync import sync_to_async
from app.core.deps import get_llm
from agent_os.knowledge.schemas import RetrievalRequest
from knowledge.models import DocumentNode

# Kích hoạt nest_asyncio để cho phép Jupyter Notebook chạy lồng loop an toàn
nest_asyncio.apply()

DIVIDER = "=" * 80


async def run_matrix_stress_test_async(questions: List[str], top_k: int):
    """
    Luồng chạy Stress Test chính thức tương thích hoàn hảo với Async Singleton Container
    """
    # Lấy Context một cách an toàn trên chính loop hiện hành của Notebook
    svc = await get_llm()

    print("\n[SYSTEM RUNTIME]")
    print(f"  RAG Khả dụng : {'AVAILABLE' if svc.rag_available else 'OFFLINE'}")
    print(f"  Cấu hình     : top_k={top_k} | threshold>={SCORE_THRESHOLD}")

    node_count = await sync_to_async(DocumentNode.objects.count)()
    print(f"  Postgres Docstore Rows: {node_count}")

    if not svc.rag_available or svc.retrieval is None:
        raise RuntimeError("RAG Infrastructure offline.")

    print(f"\n{DIVIDER}")
    print("📋 MATRIX STRESS TEST REPORT (PANDAS VIEW)")
    print(DIVIDER)

    total_dupes = 0
    summary_stats = {"RESOLVED": 0, "SAME": 0, "EMPTY": 0}

    for idx, query_text in enumerate(questions, 1):
        print(f"\n🔍 [TEST #{idx}/{len(questions)}] QUERY: {query_text}")

        req = RetrievalRequest(query=query_text, top_k=top_k)
        try:
            # Gọi trực tiếp thông qua Async Pipeline chuẩn chỉnh
            result = await svc.retrieval.retrieve(req)
        except Exception as exc:
            print(f"   ❌ LỖI HỆ THỐNG RETRIEVE VỚI QUERY NÀY: {exc}")
            import traceback

            traceback.print_exc()
            continue

        chunks = result.chunks
        dupe_indices = _detect_duplicates(chunks)
        total_dupes += len(dupe_indices)

        if not chunks:
            print("   ⚠️ Không có chunk nào vượt ngưỡng score.")
            continue

        scenario_report = []
        for ci, chunk in enumerate(chunks, 1):
            p_status = _parent_status(chunk)

            if p_status in summary_stats:
                summary_stats[p_status] += 1
            else:
                summary_stats[p_status] = 1

            status_mapping = {
                "RESOLVED": "✅ OK",
                "SAME": "🔄 SAME",
                "EMPTY": "❌ EMPTY",
            }
            p_icon = status_mapping.get(p_status, "❓ UNK")

            meta = chunk.metadata or {}

            scenario_report.append(
                {
                    "STT": ci,
                    "Score": f"{chunk.score:.4f}"
                    + (" 🔥" if ci in dupe_indices else ""),
                    "Doc ID": meta.get("source_id", "N/A"),
                    "Chunk Idx": meta.get("chunk_page_index")
                    if meta.get("chunk_page_index") is not None
                    else meta.get("chunk_index", 0),
                    "Parent Status": p_icon,
                    "Text Preview (2 Lines)": _preview_text(chunk.text, max_lines=2),
                }
            )

        df_scenario = pd.DataFrame(scenario_report).set_index("STT")
        display(df_scenario)

    print(f"\n{DIVIDER}")
    print("📊 FINAL SUMMARY REPORT")
    print(DIVIDER)
    print(f"  Tổng số câu hỏi test : {len(questions)}")
    print(f"  Trùng lặp (Dupes)    : {total_dupes} chunks")
    print(f"  Parent RESOLVED (✅) : {summary_stats.get('RESOLVED', 0)}  <- Sạch lỗi")
    print(f"  Parent SAME (🔄)     : {summary_stats.get('SAME', 0)}  <- Cần re-ingest")
    print(f"  Parent EMPTY (❌)    : {summary_stats.get('EMPTY', 0)}  <- Lỗi Docstore")
    print(f"{DIVIDER}\nDONE.")


# Khởi chạy an toàn trực tiếp dựa trên Event Loop có sẵn của Jupyter Cell
# Không tạo Thread mới để tránh phân mảnh tài nguyên và xung đột Singleton State
await run_matrix_stress_test_async(questions=QUESTIONS_TO_TEST, top_k=TOP_K)

INFO 2026-05-20 22:12:14,031 retrieval_service 915 137988097612672 [Retrieval] Khởi chạy truy xuất chuẩn LlamaIndex: query=Tổng ngân sách dự kiến mà tập đoàn cấp cho nhãn hàng EcoCare là bao nhiêu và phân bổ cụ thể cho kênh Google Search & Display Network là bao nhiêu phần trăm tổng ngân sách?


Django setup OK — 6.0.4
Settings module  : config.settings.local
Đã nạp 8 câu hỏi test hệ thống.
Cấu hình: top_k=5 | score_threshold>=0.6

[SYSTEM RUNTIME]
  RAG Khả dụng : AVAILABLE
  Cấu hình     : top_k=5 | threshold>=0.6
  Postgres Docstore Rows: 68

📋 MATRIX STRESS TEST REPORT (PANDAS VIEW)

🔍 [TEST #1/8] QUERY: Tổng ngân sách dự kiến mà tập đoàn cấp cho nhãn hàng EcoCare là bao nhiêu và phân bổ cụ thể cho kênh Google Search & Display Network là bao nhiêu phần trăm tổng ngân sách?


,Score,Doc ID,Chunk Idx,Parent Status,Text Preview (2 Lines)
STT,,,,,
1,0.5845,97,0,✅ OK,đoạn của nhãn hàng EcoCare: / - Chiến dịch thá...
2,0.5839,97,0,✅ OK,2. ĐỐI VỚI NHÃN HÀNG ECOHOME (Dòng sản phẩm cũ...
3,0.5835,94,2,✅ OK,[PHẦN 3: KHUNG GIỜ ĐẶC BIỆT CHO KHỐI VẬN HÀNH ...
4,0.5831,97,0,✅ OK,Mọi bài viết giới thiệu sản phẩm mới hoặc chiế...
5,0.5825,97,0,✅ OK,quy trình phê duyệt nội dung được chuẩn hóa th...


INFO 2026-05-20 22:12:20,819 retrieval_service 915 137988097612672 [Retrieval] Khởi chạy truy xuất chuẩn LlamaIndex: query=Vào ngày 20/07/2026, tôi có thể mua sản phẩm nước lau sàn EcoCare bản 1.5L trên hệ thống siêu thị WinMart với giá bao nhiêu?



🔍 [TEST #2/8] QUERY: Vào ngày 20/07/2026, tôi có thể mua sản phẩm nước lau sàn EcoCare bản 1.5L trên hệ thống siêu thị WinMart với giá bao nhiêu?


...

,Score,Doc ID,Chunk Idx,Parent Status,Text Preview (2 Lines)
STT,,,,,
1,0.6578,92,0,✅ OK,[BẢN CHÍNH THỨC] QUY CHẾ VÀ ĐỊNH MỨC CHI PHÍ C...
2,0.6512,96,0,✅ OK,Cụ thể: / - Nước lau sàn EcoCare (Bản 1.5L): G...
3,0.6465,91,2,✅ OK,- Giai đoạn 2 (16/07/2026 - 31/08/2026): Mở bá...
4,0.6455,97,0,✅ OK,- Giai đoạn 3 (Từ 01/09/2026 trở đi): Chính th...
5,0.6440,97,0,✅ OK,Giá bán quay về đúng giá niêm yết gốc là 350.0...


INFO 2026-05-20 22:13:05,370 retrieval_service 915 137988097612672 [Retrieval] Khởi chạy truy xuất chuẩn LlamaIndex: query=Chiến dịch quảng cáo ngoài trời (OOH) của EcoCare tại các trạm xe buýt Đà Nẵng có ngân sách cố định là bao nhiêu?



🔍 [TEST #8/8] QUERY: Chiến dịch quảng cáo ngoài trời (OOH) của EcoCare tại các trạm xe buýt Đà Nẵng có ngân sách cố định là bao nhiêu?


,Score,Doc ID,Chunk Idx,Parent Status,Text Preview (2 Lines)
STT,,,,,
1,0.5777,95,0,✅ OK,- Quy định giao ban giữa các ca: Toàn bộ nhân ...
2,0.5773,94,2,✅ OK,[PHẦN 3: KHUNG GIỜ ĐẶC BIỆT CHO KHỐI VẬN HÀNH ...
3,0.5755,91,3,✅ OK,[PHẦN 3: MA TRẬN PHÂN QUYỀN VÀ QUY TRÌNH DUYỆT...
4,0.5752,95,0,✅ OK,- Thời gian nghỉ trưa cố định: Từ 12:00 trưa đ...
5,0.5747,97,0,✅ OK,[PHẦN 3: MA TRẬN PHÂN QUYỀN VÀ QUY TRÌNH DUYỆT...



📊 FINAL SUMMARY REPORT
  Tổng số câu hỏi test : 8
  Trùng lặp (Dupes)    : 0 chunks
  Parent RESOLVED (✅) : 40  <- Sạch lỗi
  Parent SAME (🔄)     : 0  <- Cần re-ingest
  Parent EMPTY (❌)    : 0  <- Lỗi Docstore
DONE.


In [3]:
# ============================================
# JUPYER TEST — DOCUMENT LOADER SERVICE
# chạy bên trong container
# ============================================

from pathlib import Path

import trafilatura

from llama_index.core import (
    Document,
    SimpleDirectoryReader,
)


class DocumentLoaderService:
    def load_web(self, url: str) -> Document:
        downloaded = trafilatura.fetch_url(url)

        if not downloaded:
            raise ValueError(f"Cannot fetch URL: {url}")

        extracted = trafilatura.extract(downloaded)

        if not extracted:
            raise ValueError(f"Cannot extract URL: {url}")

        return Document(
            text=extracted,
            metadata={
                "source": url,
                "type": "web",
            },
        )

    def load_file(self, path: str):
        reader = SimpleDirectoryReader(input_files=[str(Path(path))])

        return reader.load_data()[0]

    def load_directory(self, path: str):
        reader = SimpleDirectoryReader(
            input_dir=str(Path(path)),
            recursive=True,
        )

        return reader.load_data()


# ============================================
# TEST
# ============================================

service = DocumentLoaderService()

# test txt/pdf/docx...
doc = service.load_file("/app/ai_stack/file/test.txt")

print("=== METADATA ===")
print(doc.metadata)

print("\n=== CONTENT ===")
print(doc.text[:1000])

ValueError: File /app/ai_stack/file/test.txt does not exist.

In [14]:
# ============================================
# FULL TEST — DOCUMENT LOADER SERVICE
# ============================================

service = DocumentLoaderService()

# ============================================
# SINGLE FILE TEST
# ============================================

test_files = [
    "/app/ai_stack/file/test.txt",
    "/app/ai_stack/file/test_docs/sample.txt",
    "/app/ai_stack/file/test_docs/sample.json",
    "/app/ai_stack/file/test_docs/sample.csv",
    "/app/ai_stack/file/test_docs/sample.md",
    "/app/ai_stack/file/test_docs/sample.html",
    "/app/ai_stack/file/test_docs/sample.xml",
    "/app/ai_stack/file/test_docs/sample.yaml",
    "/app/ai_stack/file/test_docs/sample.py",
    "/app/ai_stack/file/test_docs/sample.log",
    "/app/ai_stack/file/test_docs/sample.ipynb",
    "/app/ai_stack/file/test_docs/sample.pdf",
    "/app/ai_stack/file/test_docs/sample.docx",
]

for file_path in test_files:
    print("\n")
    print("=" * 60)
    print(f"TEST FILE: {file_path}")
    print("=" * 60)

    try:
        doc = service.load_file(file_path)

        print("\n=== METADATA ===")
        print(doc.metadata)

        print("\n=== CONTENT ===")
        print(doc.text[:1000])

    except Exception as e:
        print(f"\nERROR: {e}")

# ============================================
# DIRECTORY TEST
# ============================================

print("\n")
print("=" * 60)
print("DIRECTORY TEST")
print("=" * 60)

docs = service.load_directory("/app/ai_stack/file/test_docs")

print(f"\nLoaded {len(docs)} docs")

for d in docs:
    print("\n-------------------")

    metadata = d.metadata

    print(f"FILE: {metadata.get('file_name')}")
    print(f"TYPE: {metadata.get('file_type')}")
    print(f"SIZE: {metadata.get('file_size')}")

    print("\nCONTENT PREVIEW:")
    print(d.text[:300])



TEST FILE: /app/ai_stack/file/test.txt

=== METADATA ===
{'file_path': '/app/ai_stack/file/test.txt', 'file_name': 'test.txt', 'file_type': 'text/plain', 'file_size': 10, 'creation_date': '2026-05-18', 'last_modified_date': '2026-05-18'}

=== CONTENT ===
xion chào


TEST FILE: /app/ai_stack/file/test_docs/sample.txt

=== METADATA ===
{'file_path': '/app/ai_stack/file/test_docs/sample.txt', 'file_name': 'sample.txt', 'file_type': 'text/plain', 'file_size': 19, 'creation_date': '2026-05-18', 'last_modified_date': '2026-05-18'}

=== CONTENT ===
Xin chào TXT file



TEST FILE: /app/ai_stack/file/test_docs/sample.json

=== METADATA ===
{'file_path': '/app/ai_stack/file/test_docs/sample.json', 'file_name': 'sample.json', 'file_type': 'application/json', 'file_size': 48, 'creation_date': '2026-05-18', 'last_modified_date': '2026-05-18'}

=== CONTENT ===
{
  "name": "Duoc",
  "message": "Hello JSON"
}



TEST FILE: /app/ai_stack/file/test_docs/sample.csv

=== METADATA ===
{'file_path': '/app

In [16]:
# Cell 3 — Boot engine (async)

# Lấy embed engine từ LLMFactory
runtime = build_runtime()
embed_engine = runtime["llm_factory"].get_embedding("default_embed")

print(f"Embed engine  : {embed_engine.__class__.__name__}")
print(f"Embed model   : {embed_engine.model}")
print(f"Embed base_url: {embed_engine.base_url}")

# Khởi tạo KnowledgeEngine — kết nối PGVector, đăng ký embedding adapter
engine = await KnowledgeEngine.create(embed_engine=embed_engine)

print(f"KnowledgeEngine ready")
print(f"PGVector table : {engine.settings.vector_table}")
print(f"Embed dim      : {engine.settings.embed_dim}")
print(f"Hybrid search  : {engine.settings.hybrid_search}")

Embed engine  : LiteEmbeddingEngine
Embed model   : ollama/all-minilm
Embed base_url: http://192.168.101.18:11434
KnowledgeEngine ready
PGVector table : knowledge_vectors
Embed dim      : 384
Hybrid search  : True


In [17]:
# Cell 4 — Smoke test embed (raw)

test_vectors = await embed_engine.embed(["Kiểm tra kết nối embedding."])

assert len(test_vectors) == 1, "Phải trả về đúng 1 vector"
assert len(test_vectors[0]) == engine.settings.embed_dim, (
    f"Dimension mismatch: got {len(test_vectors[0])}, "
    f"expected {engine.settings.embed_dim}"
)

print(
    f"Embedding OK — dim={len(test_vectors[0])} (first 5 values: {test_vectors[0][:5]})"
)

Embedding OK — dim=384 (first 5 values: [-0.02649936, -0.020429857, 0.06490229, -0.01424854, 0.032250725])


In [20]:
# Cell 5 — Thám tử tìm tên bảng thực tế trong Postgres
import os

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

from django.db import connection


def find_real_vector_table():
    with connection.cursor() as cursor:
        try:
            # 1. Lấy tất cả các tên bảng hiện có trong DB công cộng (public schema)
            cursor.execute("""
                SELECT table_name 
                FROM information_schema.tables 
                WHERE table_schema = 'public'
                ORDER BY table_name;
            """)
            tables = [row[0] for row in cursor.fetchall()]

            print("📋 --- DANH SÁCH CÁC BẢNG ĐANG CÓ TRONG DB CỦA BẠN ---")
            vector_tables = []
            for t in tables:
                print(f"  • {t}")
                # Tiện tay lọc ra các bảng có từ khóa nghi vấn
                if "vector" in t or "knowledge" in t or "embed" in t or "chunk" in t:
                    vector_tables.append(t)

            print("\n🔍 --- CÁC BẢNG CÓ KHẢ NĂNG LÀ BẢNG VECTOR ---")
            if vector_tables:
                for vt in vector_tables:
                    # Đếm thử xem bảng nghi vấn đó có dữ liệu không
                    cursor.execute(f"SELECT COUNT(*) FROM {vt};")
                    count = cursor.fetchone()[0]
                    print(f"  👉 Bảng: '{vt}' ➔ Đang có {count} dòng.")
            else:
                print("  ❌ Không tìm thấy bảng nào chứa từ khóa vector/knowledge cả!")

        except Exception as e:
            print(f"❌ Lỗi hệ thống: {e}")


find_real_vector_table()

📋 --- DANH SÁCH CÁC BẢNG ĐANG CÓ TRONG DB CỦA BẠN ---
  • account_emailaddress
  • account_emailconfirmation
  • ai_engine_chatmessage
  • ai_engine_chatsession
  • ai_engine_userdocument
  • ai_memory_chunks
  • auth_group
  • auth_group_permissions
  • auth_permission
  • checkpoint_blobs
  • checkpoint_migrations
  • checkpoint_writes
  • checkpoints
  • data_ai_memory
  • data_knowledge_memory_v1
  • data_knowledge_vault_single_model
  • data_knowledge_vault_v1
  • data_knowledge_vectors
  • django_admin_log
  • django_celery_beat_clockedschedule
  • django_celery_beat_crontabschedule
  • django_celery_beat_intervalschedule
  • django_celery_beat_periodictask
  • django_celery_beat_periodictasks
  • django_celery_beat_solarschedule
  • django_content_type
  • django_migrations
  • django_session
  • django_site
  • knowledge_documentsource
  • mfa_authenticator
  • socialaccount_socialaccount
  • socialaccount_socialapp
  • socialaccount_socialapp_sites
  • socialaccount_socialtoke

In [7]:
# Cell 5 — Ingest documents

ingest_svc = IngestionService(engine=engine)

docs = [
    IngestRequest(
        text="Công ty làm việc lúc 8h30. Đi muộn quá 3 lần sẽ bị trừ KPI.",
        metadata={"source": "internal_rule", "category": "hr_policy"},
    ),
    IngestRequest(
        text="Nhân viên được nghỉ 12 ngày phép mỗi năm. Phép năm không tích luỹ sang năm sau.",
        metadata={"source": "internal_rule", "category": "hr_policy"},
    ),
    IngestRequest(
        text="Quy trình onboarding: tuần 1 học văn hoá công ty, tuần 2 shadow mentor, tuần 3 nhận task thực.",
        metadata={"source": "internal_rule", "category": "onboarding"},
    ),
]

for doc in docs:
    result = await ingest_svc.ingest(doc)
    print(
        f"[{result.status}] doc_id={result.doc_id} | text snippet: '{doc.text[:50]}...'"
    )

[success] doc_id=51397750-8f54-426a-88d0-d92f1cbaf153 | text snippet: 'Công ty làm việc lúc 8h30. Đi muộn quá 3 lần sẽ bị...'
[success] doc_id=fe751463-b0a8-49a3-832e-b6431eb5a97f | text snippet: 'Nhân viên được nghỉ 12 ngày phép mỗi năm. Phép năm...'
[success] doc_id=0dfb855e-0337-4f6e-ba63-29f20bbe5aa0 | text snippet: 'Quy trình onboarding: tuần 1 học văn hoá công ty, ...'


In [3]:
# Cell 6 — Retrieve


retrieval_svc = RetrievalService(engine=engine)

result = await retrieval_svc.retrieve(
    RetrievalRequest(
        query="Chuyển toàn bộ hệ thống sang async pipeline",
        top_k=3,
    )
)

print(f"Query : {result.query}")
print(f"Chunks: {len(result.chunks)} returned\n")

for i, chunk in enumerate(result.chunks, 1):
    print(f"--- Chunk #{i} ---")
    print(f"  score   : {chunk.score:.4f}")
    print(f"  metadata: {chunk.metadata}")
    print(f"  text    : {chunk.text[:120]}")
    print()

NameError: name 'RetrievalService' is not defined

In [16]:
import asyncio
from agent_os.system.knowledge.retrieval_service import RetrievalService
from agent_os.system.knowledge.schemas import RetrievalRequest

# 1. Khởi tạo cụm service cô lập
services = await create_isolated_services()

# 2. Gọi thẳng thuộc tính ra bằng dấu chấm
# Thường sẽ là .retrieval_service hoặc .retrieval
retrieval_service = getattr(services, "retrieval_service", None) or getattr(
    services, "retrieval", None
)

if not retrieval_service:
    # Nếu dùng getattr phòng hờ không ra, cậu in thử các thuộc tính đang có để nhìn cho rõ:
    print(
        "Các dịch vụ có trong AgentServices của cậu:",
        [attr for attr in dir(services) if not attr.startswith("_")],
    )
    # Gán trực tiếp bằng tay nếu biết tên: ví dụ retrieval_service = services.retrieval_service


# 3. Định nghĩa hàm test
async def check_retrieval(query_str: str, top_k: int = 3):
    print(f"🔍 Đang tìm kiếm cụm từ: '{query_str}'...\n")
    request = RetrievalRequest(query=query_str, top_k=top_k)
    result = await retrieval_service.retrieve(request)

    print(f"📊 Kết quả cho câu truy vấn: '{result.query}'")
    print(f"📁 Số lượng chunks vượt qua bộ lọc (Score >= 0.45): {len(result.chunks)}")
    print("-" * 60)

    if not result.chunks:
        print(
            "❌ Không tìm thấy đoạn văn bản nào phù hợp hoặc điểm score quá thấp (< 0.45)."
        )

    for idx, chunk in enumerate(result.chunks, 1):
        print(f"[{idx}] [Score: {chunk.score:.4f}]")
        print(f"📄 Nội dung: {chunk.text.strip()}")
        print("-" * 60)


# 4. Chạy check
await check_retrieval(query_str="Tối ưu chunking strategy", top_k=3)

🔍 Đang tìm kiếm cụm từ: 'Tối ưu chunking strategy'...

📊 Kết quả cho câu truy vấn: 'Tối ưu chunking strategy'
📁 Số lượng chunks vượt qua bộ lọc (Score >= 0.45): 0
------------------------------------------------------------
❌ Không tìm thấy đoạn văn bản nào phù hợp hoặc điểm score quá thấp (< 0.45).


In [9]:
# Cell 7 — Assertions (pass/fail)


# Assertion 1: phải có kết quả trả về
assert len(result.chunks) > 0, "Retrieval phải trả về ít nhất 1 chunk"

# Assertion 2: số chunk không vượt quá top_k đã yêu cầu
assert len(result.chunks) <= 3, f"Trả về nhiều hơn top_k=3: {len(result.chunks)}"

# Assertion 3: score phải là số thực hợp lệ, không phải None
for i, chunk in enumerate(result.chunks):
    assert isinstance(chunk.score, float), (
        f"Chunk #{i} score phải là float, got {type(chunk.score)}"
    )

# Assertion 4: metadata phải còn nguyên sau vòng ingest → retrieve
sources = {chunk.metadata.get("source") for chunk in result.chunks}
assert "internal_rule" in sources, "Metadata 'source' bị mất sau khi ingest/retrieve"

# Assertion 5: text không được rỗng
for i, chunk in enumerate(result.chunks):
    assert chunk.text.strip(), f"Chunk #{i} có text rỗng"

# Info: in ra ranking thực tế để quan sát chất lượng embedding
print("Ranking thực tế từ model embedding:")
for i, chunk in enumerate(result.chunks, 1):
    print(f"  #{i} score={chunk.score:.4f} | {chunk.text[:60]}...")

print("\nAll assertions passed — RAG pipeline OK")

Ranking thực tế từ model embedding:
  #1 score=0.5083 | Công ty làm việc lúc 8h30. Đi muộn quá 3 lần sẽ bị trừ KPI....
  #2 score=0.5083 | Công ty làm việc lúc 8h30. Đi muộn quá 3 lần sẽ bị trừ KPI....

All assertions passed — RAG pipeline OK


In [10]:
# Cell 8 — Cleanup (tuỳ chọn)

# Bỏ comment dòng dưới để xoá toàn bộ vector table

# engine.vector_store.close()
# import asyncpg
# conn = await asyncpg.connect(
#     host=engine.settings.postgres_host,
#     port=engine.settings.postgres_port,
#     user=engine.settings.postgres_user,
#     password=engine.settings.postgres_password,
#     database=engine.settings.postgres_db,
# )
# await conn.execute(f'DROP TABLE IF EXISTS "{engine.settings.vector_table}" CASCADE')
# await conn.close()
# print(f"Table '{engine.settings.vector_table}' dropped.")

print("Cleanup cell — bỏ comment nếu cần xoá bảng test.")

Cleanup cell — bỏ comment nếu cần xoá bảng test.


In [2]:
# Cell 6 — Retrieve & Test Vector DB Connection (Đã sửa tên bảng chuẩn)
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import asyncio
from llama_index.core import Settings
from llama_index.embeddings.ollama import OllamaEmbedding
from agent_os.system.runtime.llm_model import llm_model
from agent_os.system.knowledge.schemas import RetrievalRequest


async def run_retrieval_test():
    # 1. Cấu hình đồng bộ Model nhúng
    Settings.embed_model = OllamaEmbedding(
        model_name="all-minilm", base_url="http://192.168.101.18:11434"
    )

    # 2. Gọi service engine của hệ thống
    services = await llm_model()

    if not services.rag_available:
        print("❌ KnowledgeEngine chưa sẵn sàng.")
        return

    # 🔥 HACK TÊN BẢNG CHUẨN: Ép Engine đọc đúng bảng chứa 13 dòng dữ liệu
    if hasattr(services.knowledge_engine, "_vector_store"):
        v_store = services.knowledge_engine._vector_store
        # Thay đổi tên bảng lưu trữ thực tế từ cấu hình sai thành cấu hình đúng
        if hasattr(v_store, "table_name"):
            v_store.table_name = "data_knowledge_vectors"
            print("🎯 Đã bẫy ép LlamaIndex đọc bảng thực tế: 'data_knowledge_vectors'")

    # 3. Lấy dịch vụ truy vấn
    retrieval_svc = services.retrieval

    # 4. Thực hiện truy vấn từ khóa "Ngày họp" hoặc "Ollama"
    print("🔍 Đang gửi câu hỏi tìm kiếm tới Vector DB thực...")
    result = await retrieval_svc.retrieve(
        RetrievalRequest(
            query="Chuyển toàn bộ hệ thống sang async pipeline",
            top_k=3,
        )
    )

    print("\n==================================================")
    print(f"Query : {result.query}")
    print(f"Chunks: {len(result.chunks)} returned\n")

    for i, chunk in enumerate(result.chunks, 1):
        print(f"--- Chunk #{i} ---")
        print(f"  score   : {chunk.score:.4f}")
        print(f"  metadata: {chunk.metadata}")
        print(f"  text    : {chunk.text[:150]}...")
        print()


# Chạy test
await run_retrieval_test()

🔍 Đang gửi câu hỏi tìm kiếm tới Vector DB thực...

Query : Chuyển toàn bộ hệ thống sang async pipeline
Chunks: 0 returned



In [ ]:
# =============================================================================
# JUPYTER CELL: FIX NGUỒN ENV & XEM TRỰC TIẾP DỮ LIỆU PGVECTOR
# =============================================================================
import os
import sys
import pathlib
import environ
import asyncio
import asyncpg
import pandas as pd

# 1. Định vị Project Root (Thư mục chứa file manage.py và file .env)
PROJECT_ROOT = pathlib.Path(".").resolve().parent
if not (PROJECT_ROOT / "agent_os").exists():
    if os.path.exists("/app"):
        PROJECT_ROOT = pathlib.Path("/app")
    elif os.path.exists("/app/marketing_crew"):
        PROJECT_ROOT = pathlib.Path("/app/marketing_crew")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 2. ÉP NẠP FILE .ENV VÀO HỆ ĐIỀU HÀNH (Giải quyết lỗi nghẽn của environ.Env())
env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    environ.Env.read_env(str(env_file))
    print(f"✅ Đã đồng bộ cấu hình môi trường từ: {env_file}")
else:
    print(
        "⚠️ Không tìm thấy file .env tại root, hệ thống sẽ dùng biến môi trường Docker hiện tại."
    )

# 3. IMPORT AN TOÀN SETTINGS HẠ TẦNG
from agent_os.system.knowledge.settings import KnowledgeSettings
import sys

sys.path.append("/app")

import nest_asyncio

nest_asyncio.apply()

import os
import json
import tempfile
import requests
import trafilatura
import pandas as pd

from docx import Document as DocxDocument
from pypdf import PdfWriter

from llama_index.core import Document
from agent_os.knowledge.knowledge_config import DocType
from agent_os.knowledge.knowledge_engine import KnowledgeEngine


async def full_system_test():

    print("\n🚀 INIT KNOWLEDGE ENGINE...")
    engine = KnowledgeEngine()

    # =====================================================
    # 1. BASIC RAG TEST
    # =====================================================
    print("\n==============================")
    print("1️⃣ BASIC RAG TEST")
    print("==============================")

    doc_mkt = Document(
        text="Chiến lược Marketing Tết 2026 tập trung vào video ngắn trên TikTok.",
        metadata={"source": "plan_tet.pdf"},
    )

    doc_legal = Document(
        text="Hợp đồng đại lý cấp 1 yêu cầu doanh số tối thiểu 1 tỷ đồng/tháng.",
        metadata={"source": "contract_v1.pdf"},
    )

    engine.ingest([doc_mkt], "alex", DocType.MARKETING.value)
    engine.ingest([doc_legal], "alex", DocType.LEGAL.value)

    # Marketing retrieve
    retriever_mkt = engine.get_retriever("alex", DocType.MARKETING.value, top_k=3)

    raw_res = retriever_mkt.retrieve("Doanh số bao nhiêu?")

    res_mkt = [r for r in raw_res if r.score and r.score > 0.7]

    print(f"\nMarketing isolation: {len(res_mkt)} kết quả")

    # Legal retrieve
    retriever_legal = engine.get_retriever("alex", DocType.LEGAL.value, top_k=3)

    res_legal = retriever_legal.retrieve("Doanh số")

    print(f"Legal retrieve: {len(res_legal)} kết quả")

    if res_legal:
        print("Sample:", res_legal[0].text[:100])

    # User isolation
    retriever_hacker = engine.get_retriever(
        "hacker_007", DocType.MARKETING.value, top_k=3
    )

    res_hacker = retriever_hacker.retrieve("TikTok")

    print(f"Hacker isolation: {len(res_hacker)} kết quả")

    # =====================================================
    # 2. WEB TEST
    # =====================================================
    print("\n==============================")
    print("2️⃣ WEB LOADER TEST")
    print("==============================")

    try:
        web_doc = engine.load_web("https://hhtm.net/kiem-lai-phan-2/tap-24")

        if web_doc and web_doc.text:
            print("✅ WEB LOAD OK")
            print("Length:", len(web_doc.text))
            print(web_doc.text[:300])
        else:
            print("❌ WEB LOAD FAIL")

    except Exception as e:
        print("❌ WEB ERROR:", e)

    # =====================================================
    # 3. TXT TEST
    # =====================================================
    print("\n==============================")
    print("3️⃣ TXT LOADER TEST")
    print("==============================")

    try:
        txt_path = "/tmp/test.txt"

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("Đây là file TXT test cho hệ thống RAG.")

        txt_doc = engine.load_txt(txt_path)

        print("✅ TXT OK")
        print(txt_doc.text)

    except Exception as e:
        print("❌ TXT ERROR:", e)

    # =====================================================
    # 4. JSON TEST
    # =====================================================
    print("\n==============================")
    print("4️⃣ JSON LOADER TEST")
    print("==============================")

    try:
        json_path = "/tmp/test.json"

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump({"name": "Alex", "role": "Marketing"}, f)

        json_doc = engine.load_json(json_path)

        print("✅ JSON OK")
        print(json_doc.text[:200])

    except Exception as e:
        print("❌ JSON ERROR:", e)

    # =====================================================
    # 5. DOCX TEST
    # =====================================================
    print("\n==============================")
    print("5️⃣ DOCX LOADER TEST")
    print("==============================")

    try:
        docx_path = "/tmp/test.docx"

        docx_file = DocxDocument()
        docx_file.add_paragraph("Đây là nội dung Word test.")
        docx_file.save(docx_path)

        docx_doc = engine.load_docx(docx_path)

        print("✅ DOCX OK")
        print(docx_doc.text)

    except Exception as e:
        print("❌ DOCX ERROR:", e)

    # =====================================================
    # 6. EXCEL TEST
    # =====================================================
    print("\n==============================")
    print("6️⃣ EXCEL LOADER TEST")
    print("==============================")

    try:
        excel_path = "/tmp/test.xlsx"

        df = pd.DataFrame({"Tên": ["Alex", "John"], "Role": ["Marketing", "Legal"]})

        df.to_excel(excel_path, index=False)

        excel_doc = engine.load_excel(excel_path)

        print("✅ EXCEL OK")
        print(excel_doc.text[:300])

    except Exception as e:
        print("❌ EXCEL ERROR:", e)

    # =====================================================
    # 7. PDF TEST
    # =====================================================
    print("\n==============================")
    print("7️⃣ PDF LOADER TEST")
    print("==============================")

    try:
        pdf_path = "/tmp/test.pdf"

        writer = PdfWriter()
        writer.add_blank_page(width=300, height=300)

        with open(pdf_path, "wb") as f:
            writer.write(f)

        pdf_doc = engine.load_pdf(pdf_path)

        print("✅ PDF OK")
        print("Length:", len(pdf_doc.text))

    except Exception as e:
        print("❌ PDF ERROR:", e)

    print("\n🎉 FULL SYSTEM TEST DONE")


await full_system_test()


async def inspect_vectors_now():
    print(
        f"🔍 Đang chọc vào Postgres: {KnowledgeSettings.postgres_host}:{KnowledgeSettings.postgres_port}"
    )
    print(f"📦 Bảng mục tiêu: '{KnowledgeSettings.vector_table}'")

    # Kết nối trực tiếp dựa trên cấu hình hạ tầng thuần của cậu
    conn = await asyncpg.connect(
        host=KnowledgeSettings.postgres_host,
        port=KnowledgeSettings.postgres_port,
        user=KnowledgeSettings.postgres_user,
        password=KnowledgeSettings.postgres_password,
        database=KnowledgeSettings.postgres_db,
    )

    try:
        # Truy vấn lôi text thô và metadata ra
        query_sql = (
            f'SELECT id, content, metadata FROM "{KnowledgeSettings.vector_table}"'
        )
        rows = await conn.fetch(query_sql)

        if not rows:
            print("❌ BẢNG ĐANG TRỐNG RỖNG! Không có bản ghi nào.")
            return

        db_data = []
        for r in rows:
            db_data.append(
                {
                    "ID": r["id"],
                    "Text Content": r["content"],
                    "Source": r["metadata"].get("source") if r["metadata"] else None,
                    "Category": r["metadata"].get("category")
                    if r["metadata"]
                    else None,
                }
            )

        df = pd.DataFrame(db_data)
        print(f"\n🎯 ĐANG LƯU TRỮ {len(df)} VECTOR TRI THỨC:")
        return df

    except Exception as e:
        print(f"❌ Lỗi truy vấn SQL: {e}")
    finally:
        await conn.close()


# Thực thi hiển thị bảng dữ liệu
await inspect_vectors_now()

In [1]:
import sys

sys.path.append("/app")

import nest_asyncio

nest_asyncio.apply()

import os
import json
import tempfile
import requests
import trafilatura
import pandas as pd

from docx import Document as DocxDocument
from pypdf import PdfWriter

from llama_index.core import Document
from agent_os.knowledge.knowledge_config import DocType
from agent_os.knowledge.knowledge_engine import KnowledgeEngine


async def full_system_test():

    print("\n🚀 INIT KNOWLEDGE ENGINE...")
    engine = KnowledgeEngine()

    # =====================================================
    # 1. BASIC RAG TEST
    # =====================================================
    print("\n==============================")
    print("1️⃣ BASIC RAG TEST")
    print("==============================")

    doc_mkt = Document(
        text="Chiến lược Marketing Tết 2026 tập trung vào video ngắn trên TikTok.",
        metadata={"source": "plan_tet.pdf"},
    )

    doc_legal = Document(
        text="Hợp đồng đại lý cấp 1 yêu cầu doanh số tối thiểu 1 tỷ đồng/tháng.",
        metadata={"source": "contract_v1.pdf"},
    )

    engine.ingest([doc_mkt], "alex", DocType.MARKETING.value)
    engine.ingest([doc_legal], "alex", DocType.LEGAL.value)

    # Marketing retrieve
    retriever_mkt = engine.get_retriever("alex", DocType.MARKETING.value, top_k=3)

    raw_res = retriever_mkt.retrieve("Doanh số bao nhiêu?")

    res_mkt = [r for r in raw_res if r.score and r.score > 0.7]

    print(f"\nMarketing isolation: {len(res_mkt)} kết quả")

    # Legal retrieve
    retriever_legal = engine.get_retriever("alex", DocType.LEGAL.value, top_k=3)

    res_legal = retriever_legal.retrieve("Doanh số")

    print(f"Legal retrieve: {len(res_legal)} kết quả")

    if res_legal:
        print("Sample:", res_legal[0].text[:100])

    # User isolation
    retriever_hacker = engine.get_retriever(
        "hacker_007", DocType.MARKETING.value, top_k=3
    )

    res_hacker = retriever_hacker.retrieve("TikTok")

    print(f"Hacker isolation: {len(res_hacker)} kết quả")

    # =====================================================
    # 2. WEB TEST
    # =====================================================
    print("\n==============================")
    print("2️⃣ WEB LOADER TEST")
    print("==============================")

    try:
        web_doc = engine.load_web("https://hhtm.net/kiem-lai-phan-2/tap-24")

        if web_doc and web_doc.text:
            print("✅ WEB LOAD OK")
            print("Length:", len(web_doc.text))
            print(web_doc.text[:300])
        else:
            print("❌ WEB LOAD FAIL")

    except Exception as e:
        print("❌ WEB ERROR:", e)

    # =====================================================
    # 3. TXT TEST
    # =====================================================
    print("\n==============================")
    print("3️⃣ TXT LOADER TEST")
    print("==============================")

    try:
        txt_path = "/tmp/test.txt"

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("Đây là file TXT test cho hệ thống RAG.")

        txt_doc = engine.load_txt(txt_path)

        print("✅ TXT OK")
        print(txt_doc.text)

    except Exception as e:
        print("❌ TXT ERROR:", e)

    # =====================================================
    # 4. JSON TEST
    # =====================================================
    print("\n==============================")
    print("4️⃣ JSON LOADER TEST")
    print("==============================")

    try:
        json_path = "/tmp/test.json"

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump({"name": "Alex", "role": "Marketing"}, f)

        json_doc = engine.load_json(json_path)

        print("✅ JSON OK")
        print(json_doc.text[:200])

    except Exception as e:
        print("❌ JSON ERROR:", e)

    # =====================================================
    # 5. DOCX TEST
    # =====================================================
    print("\n==============================")
    print("5️⃣ DOCX LOADER TEST")
    print("==============================")

    try:
        docx_path = "/tmp/test.docx"

        docx_file = DocxDocument()
        docx_file.add_paragraph("Đây là nội dung Word test.")
        docx_file.save(docx_path)

        docx_doc = engine.load_docx(docx_path)

        print("✅ DOCX OK")
        print(docx_doc.text)

    except Exception as e:
        print("❌ DOCX ERROR:", e)

    # =====================================================
    # 6. EXCEL TEST
    # =====================================================
    print("\n==============================")
    print("6️⃣ EXCEL LOADER TEST")
    print("==============================")

    try:
        excel_path = "/tmp/test.xlsx"

        df = pd.DataFrame({"Tên": ["Alex", "John"], "Role": ["Marketing", "Legal"]})

        df.to_excel(excel_path, index=False)

        excel_doc = engine.load_excel(excel_path)

        print("✅ EXCEL OK")
        print(excel_doc.text[:300])

    except Exception as e:
        print("❌ EXCEL ERROR:", e)

    # =====================================================
    # 7. PDF TEST
    # =====================================================
    print("\n==============================")
    print("7️⃣ PDF LOADER TEST")
    print("==============================")

    try:
        pdf_path = "/tmp/test.pdf"

        writer = PdfWriter()
        writer.add_blank_page(width=300, height=300)

        with open(pdf_path, "wb") as f:
            writer.write(f)

        pdf_doc = engine.load_pdf(pdf_path)

        print("✅ PDF OK")
        print("Length:", len(pdf_doc.text))

    except Exception as e:
        print("❌ PDF ERROR:", e)

    print("\n🎉 FULL SYSTEM TEST DONE")


await full_system_test()

ModuleNotFoundError: No module named 'agent_os.knowledge'